In [ ]:
# Running a jupyter code block will store the vars in memory
# Therefore, restart the kernel to clear memory
from prettytable import PrettyTable
from pyspark import SparkContext, SparkConf

conf = SparkConf().setMaster("local").setAppName("FakeFriends")
sc = SparkContext(conf=conf)

def parseLine(line):
    fields = line.split(',')
    age = int(fields[2])
    numFriends = int(fields[3])
    return (age, numFriends)

lines = sc.textFile("fakefriends.csv")
rdd = lines.map(parseLine)

# collect data from RDD
data = rdd.collect()

# create table
table = PrettyTable()
table.field_names = ["Age", "Number of Friends"]

for age, num_friends in data:
    table.add_row([age, num_friends])
print("Number of Friends by Age")
print(table)

# Count up sum of friends and number of entries per age
# rdd: (age, num_friends) : (22,300)
#    (22, (300, 1))
#  + (22, (200, 1))
#  = (22, (500, 2))
# &&&&&&&&&&&&&&&&&&&&
#    (44, (30, 1))
#  + (44, (32, 1))
#  = (44, (62, 2))
totalsByAge = (
    rdd.mapValues(lambda x: (x, 1))
    .reduceByKey(lambda x, y: (x[0] + y[0], x[1] + y[1]))
)
# If access to key is not needed, use mapValues for better performance.

# This Rdd isn't be used for anything since we calculate it manually later
# we could do a join on the two RDDs then print from combined result
averageByAge = totalsByAge.mapValues(lambda x: x[0] / x[1])
totalsData = totalsByAge.collect()
totalsTable = PrettyTable()
totalsTable.field_names = ["Age", "Total Friends", "Count", "Average Fiends"]


for age, (totalFriends, count) in sorted(totalsData):
    avgFriends = totalFriends / count
    totalsTable.add_row([age, totalFriends, count, avgFriends])

print("\nTotals by Age")
print(totalsTable)

# run code, then stop spark context to free up memory
sc.stop()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/29 14:16:45 WARN Utils: Your hostname, AsusLaptop, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/06/29 14:16:45 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/29 14:16:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Number of Friends by Age
+-----+-------------------+
| Age | Number of Friends |
+-----+-------------------+
|  33 |        385        |
|  33 |         2         |
|  55 |        221        |
|  40 |        465        |
|  68 |         21        |
|  17 |         1         |
+-----+-------------------+

Totals by Age
+-----+---------------+-------+----------------+
| Age | Total Friends | Count | Average Fiends |
+-----+---------------+-------+----------------+
|  17 |       1       |   1   |      1.0       |
|  33 |      387      |   2   |     193.5      |
|  40 |      465      |   1   |     465.0      |
|  55 |      221      |   1   |     221.0      |
|  68 |       21      |   1   |      21.0      |
+-----+---------------+-------+----------------+


In [1]:
from pyspark import SparkConf, SparkContext
from prettytable import PrettyTable

conf = SparkConf().setMaster('local').setAppName('MinTemps')
sc = SparkContext(conf = conf)

def parseLine(line):
    fields = line.split(',')
    stationID = fields[0]
    entryType = fields[2]
    temp = float(fields[3])
    return (stationID, entryType, temp)

lines = sc.textFile('weatherData1800s.csv')
parsedLines = lines.map(parseLine)
minTemps = parsedLines.filter(lambda x: 'TMIN' in x[1])
stationTemps = minTemps.map(lambda x: (x[0], x[2]))
minTemps = stationTemps.reduceByKey(lambda x,y: min(x,y))
results = minTemps.collect()

table = PrettyTable()
table.field_names = ["Station ID", "Temp"]

for (id, temp) in results:
    table.add_row([id, temp])

print("minTemps")
print(table)

""" ==========================================================
Challenge: Find max temps
========================================================== """
maxTemps = parsedLines.filter(lambda x: 'TMAX' in x[1])\
    .map(lambda x: (x[0], x[2]))\
    .reduceByKey(max)
# result_temps = maxTemps.reduceByKey(lambda x,y: max(x[1],y[1]))
results = maxTemps.collect()

max_table = PrettyTable()
max_table.field_names = ["Station ID", "Temp"]

for (id, temp) in results:
    max_table.add_row([id, temp])

print("maxTemps")
print(max_table)

# run code, then stop spark context to free up memory
sc.stop()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/29 15:07:31 WARN Utils: Your hostname, AsusLaptop, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/06/29 15:07:31 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/29 15:07:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


minTemps
+-------------+--------+
|  Station ID |  Temp  |
+-------------+--------+
| ITE00100554 | -148.0 |
| EZE00100082 | -135.0 |
+-------------+--------+
maxTemps
+-------------+-------+
|  Station ID |  Temp |
+-------------+-------+
| ITE00100554 | 323.0 |
| EZE00100082 | 323.0 |
+-------------+-------+
